Copyright (c) Meta Platforms, Inc. and affiliates.

<a target="_blank" href="https://colab.research.google.com/github/facebookresearch/co-tracker/blob/main/notebooks/demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# CoTracker: It is Better to Track Together
This is a demo for <a href="https://co-tracker.github.io/">CoTracker</a>, a model that can track any point in a video.

Don't forget to turn on GPU support if you're running this demo in Colab. 

**Runtime** -> **Change runtime type** -> **Hardware accelerator** -> **GPU**

Let's install dependencies for Colab:

In [ ]:
%cd ..
%load_ext autoreload
%autoreload 2
import os
import torch

from base64 import b64encode
from cotracker.utils.visualizer import Visualizer, read_video_from_path, read_images_from_path
from IPython.display import HTML

def show_video(video_path):
    video_file = open(video_path, "r+b").read()
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    return HTML(f"""<video width="1280" height="960" autoplay loop controls><source src="{video_url}"></video>""")

Read a video

In [ ]:
asset_folder = './assets'
img_folder_name = 'maila/overlay'

img_folder_path = os.path.join(asset_folder, img_folder_name)


video = read_images_from_path(img_folder_path)
video = torch.from_numpy(video).permute(0, 3, 1, 2)[None].float() # video should be of shape (1, B, C, H, W), with B is number of frames
print(f"Video shape after reshape: {video.shape}")

In [ ]:
import time
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import torch
import numpy as np

# Squeeze the first dimension (batch size)
video_tensor = video.squeeze(0)

# Convert the tensor to numpy for matplotlib
video_tensor = video_tensor.permute(0, 2, 3, 1).numpy()  # Shape (B, H, W, C)

# Create a figure for displaying
fig, ax = plt.subplots()

# Loop through frames and display them
for frame in video_tensor[:10]:
    ax.imshow(frame.astype(np.uint8))
    display(fig)  # Display the figure in the notebook cell
    clear_output(wait=True)  # Clear the output to show the next frame
    plt.pause(0.1)  # Pause to simulate video frame display

plt.show()

Import CoTrackerPredictor and create an instance of it. We'll use this object to estimate tracks:

## Manuall select points for all frames

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import random
from typing import List, Optional, Sequence, Tuple
import colorsys


# From https://github.com/google-deepmind/tapnet/blob/main/tapnet/utils/viz_utils.py
# Generate random colormaps for visualizing different points.
def get_colors(num_colors: int) -> List[Tuple[int, int, int]]:
  """Gets colormap for points."""
  colors = []
  for i in np.arange(0.0, 360.0, 360.0 / num_colors):
    hue = i / 360.0
    lightness = (50 + np.random.rand() * 10) / 100.0
    saturation = (90 + np.random.rand() * 10) / 100.0
    color = colorsys.hls_to_rgb(hue, lightness, saturation)
    colors.append(
        (int(color[0] * 255), int(color[1] * 255), int(color[2] * 255))
    )
  random.shuffle(colors)
  return colors

video = video.squeeze(0)
select_frame = 0  # Starting frame (image)
num_frames = len(video)  # Total number of frames in the video (or images)
colormap = get_colors(50)

# Initialize the figure and axis
fig, ax = plt.subplots(figsize=(15, 15))
image_points = []
all_points = np.zeros((num_frames, 4, 2), dtype=np.int32)

def show_image(frame):
    ax.clear()  # Clear the axis
    ax.imshow(video[frame, :].permute(1, 2, 0).cpu().numpy().astype(np.uint8))
    ax.axis('off')
    ax.set_title(f'Select points for frame {frame}. Click order: HR, HL, VR, VL.')
    plt.draw()

def on_click(event):
    if event.button == 1 and event.inaxes == ax:  # Left mouse button clicked
        x, y = int(np.round(event.xdata)), int(np.round(event.ydata))
        image_points.append([x, y])
        ax.plot(x, y, 'o', color="red", markersize=5)
        plt.draw()

def next_frame(event):
    global select_frame
    
    all_points[select_frame] = np.array(image_points)
    image_points.clear()  # Clear the points for the next frame
    print(all_points)
    
    if select_frame < num_frames - 1:
        select_frame += 1
        show_image(select_frame)
    else:
        print("Reached the last frame.")

# Set up event listeners
fig.canvas.mpl_connect('button_press_event', on_click)
fig.canvas.mpl_connect('key_press_event', next_frame)  # Listen for key press to go to next frame

show_image(select_frame)  # Show the first image
plt.show()

# HL, HR, VL, VR


We pass these points as input to the model and track them:

In [ ]:
# replace all invalid selected points (ie top right corner) with zero depth values
all_points[(all_points[..., 0] > 400) & (all_points[..., 1] < 100)] = 0.0

In [ ]:
# all_points = np.load("videos/maila/overlay/foot_pos.npy", allow_pickle=True)
# all_points[28, [0, 1]] = all_points[28, [1, 0]]
# all_points[45, [0, 1]] = all_points[45, [1, 0]]
# all_points[46, [0, 1]] = all_points[46, [1, 0]]


video_path = results_path = os.path.join('./videos', img_folder_name.rsplit('.', 1)[0])
os.makedirs(video_path, exist_ok=True)
os.makedirs(os.path.join(video_path, "images_with_points"), exist_ok=True)


# Define colors for the four points
colors = ['red', 'blue', 'green', 'yellow']


# Create a figure for displaying
fig, ax = plt.subplots()

# Loop through frames and save them
for i, frame in enumerate(video_tensor):
    ax.clear()
    ax.imshow(frame.astype(np.uint8))

    # Plot the four points with different colors
    for j in range(4):
        x, y = all_points[i, j]  # Extract x and y coordinates
        ax.scatter(x, y, color=colors[j], s=50)  # Scatter plot for points

    # Save the frame
    frame_filename = os.path.join(video_path, "images_with_points", f"frame_{i:04d}.png")
    plt.savefig(frame_filename, bbox_inches='tight', pad_inches=0)

plt.close(fig)  # Close the figure to free memory


In [ ]:
np.save(os.path.join(video_path, 'foot_pos.npy'), all_points)

Finally, we visualize the results with tracks leaving traces from the frame where the tracking starts.
Color encodes time:

Notice that points queried at frames 10, 20, and 30 are tracked **incorrectly** before the query frame. This is because CoTracker is an online algorithm and only tracks points in one direction. However, we can also run it backward from the queried point to track in both directions. Let's correct this: